# dependencies

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Literal, Dict, Any, Optional
from dataclasses import dataclass
from langchain_community.tools import TavilySearchResults
from langchain_experimental.tools import PythonREPLTool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent
from langgraph.types import Command
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END, MessagesState

import os
os.makedirs("logs", exist_ok=True)
import sys
import logging
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(filename='logs/supervisor.log', mode='a')
    ]
)
logger = logging.getLogger(__name__)

from dotenv import load_dotenv
load_dotenv()

# constants and type definitions

In [ ]:
SUPERVISOR_PROMPT = """
You are a workflow supervisor managing a team of three specialized agents: Prompt Enhancer, Researcher, and Coder.
Choose the next best agent to move the user request toward completion efficiently.

Team Members:
1. Prompt Enhancer: Clarifies & restructures ambiguous or under-specified queries (consider first).
2. Researcher: Performs external factual lookups and gathers structured information only.
3. Coder: Performs computation, code execution, math, data manipulation, or algorithmic reasoning.

Responsibilities:
- Decide the next agent with minimal redundancy.
- Provide a concise justification for the routing decision.

Respond ONLY with a JSON object matching the schema.
"""

ENHANCER_PROMPT = """
You are a Query Refinement Specialist.
Tasks:
1. Disambiguate and expand vague user input using reasonable assumptions.
2. DO NOT ask the user questions.
3. Produce a clearer, actionable rewritten version of the user's intent.
Return only the refined version.
"""

RESEARCHER_STATE_MODIFIER = (
    "You are an Information Specialist.\n"
    "Goals:\n"
    "1. Identify factual info needs.\n"
    "2. Use search tools to gather up-to-date structured facts.\n"
    "3. Cite any sources when available.\n"
    "4. Do NOT speculate. Do NOT implement or analyze—just collect info.\n"
)

CODER_STATE_MODIFIER = (
    "You are a coder & analyst. Focus on calculations, algorithmic reasoning, code execution, "
    "and technical problem solving. Use tools when helpful."
)

VALIDATOR_PROMPT = """
You are a validator.
Rules:
- Compare the original user question (first message) with the current answer (last message).
- If answer addresses the core intent sufficiently (even if imperfect) => FINISH.
- Only route back to supervisor if answer is completely off-topic, harmful, or fundamentally wrong.
- Prefer finishing over loops. Accept 'good enough'.

Return JSON per schema.
"""

DEFAULT_USER_QUERY = "Weather in Mumbai"

# schema / state definitions

In [ ]:
class SupervisorDecision(BaseModel):
    next: Literal['enhancer', 'researcher', 'coder'] = Field(description="Next specialist agent to route to")
    reason: str = Field(description="Concise justification for the routing decision")

class ValidatorDecision(BaseModel):
    next: Literal['supervisor', 'FINISH'] = Field(description="Either 'FINISH' to end or 'supervisor' to re-route")
    reason: str = Field(description="Justification for the decision")

# tool definitions

In [ ]:
@dataclass
class ToolResources:
    tavily: TavilySearchResults
    python_repl: PythonREPLTool

def init_tools() -> ToolResources:
    tavily_tool = TavilySearchResults(max_results=2, search_depth='basic')
    python_tool = PythonREPLTool()
    
    try:
        python_tool.invoke('x = 5; print(x)')
    except Exception as e:
        logger.error(f"❌ Python REPL tool initialization failed: {e}")
    return ToolResources(tavily=tavily_tool, python_repl=python_tool)


# configuration model

In [ ]:
@dataclass
class WorkflowConfig:
    model_name: str = 'gemini-2.5-flash'
    temperature: float = 0.1
    max_search_results: int = 2
    supervisor_prompt: str = SUPERVISOR_PROMPT
    enhancer_prompt: str = ENHANCER_PROMPT
    validator_prompt: str = VALIDATOR_PROMPT
    researcher_state_modifier: str = RESEARCHER_STATE_MODIFIER
    coder_state_modifier: str = CODER_STATE_MODIFIER

# main class

In [ ]:
class MultiAgentWorkflow:
    def __init__(self, config: WorkflowConfig):
        self.config = config
        self.llm = self._init_llm()
        self.tools = init_tools()
        self.graph = StateGraph(MessagesState)
        self.app = None
        self._register_nodes()
        self._build_graph()

        logger.info("🚀 MultiAgentWorkflow initialized.")
    
    # llm init
    def _init_llm(self) -> ChatGoogleGenerativeAI:
        try:
            llm = ChatGoogleGenerativeAI(model=self.config.model_name, temperature=self.config.temperature)
            return llm
        except Exception as e:
            logger.error(f"❌ LLM initialization failed: {e}")
            raise e
        
    # node definitions
    def supervisor_node(self, state: MessagesState) -> Command[Literal['enhancer', 'researcher', 'coder']]:
        try:
            messages = [
                {
                    'role': 'system',
                    'content': self.config.supervisor_prompt
                }
            ] + state['messages']
            response = self.llm.with_structured_output(SupervisorDecision).invoke(messages)
            goto = response.next
            reason = response.reason
            logger.debug(f"✅ Supervisor routed to {goto} & reason: {reason}")

            return Command(
                update={'messages': [HumanMessage(content=f"Supervisor routed to {goto} because: {reason}", name='supervisor')]},
                goto=goto
            )
        except Exception as e:
            logger.error(f"❌ Supervisor node failed: {e}")
            return Command(
                update={'messages': [HumanMessage(content=f"Supervisor error fallback: {e}", name='supervisor')]},
                goto='enhancer'
            )

    def enhancer_node(self, state: MessagesState) -> Command[Literal['supervisor']]:
        try: 
            messages = [
                {
                    'role': 'system',
                    'content': self.config.enhancer_prompt
                }
            ] + state['messages']
            response = self.llm.invoke(messages)
            logger.debug(f"✨ Enhancer produced refined query: {response}")
            return Command(
                update={'messages': [HumanMessage(content=response, name='enhancer')]},
                goto='supervisor'
            )
        except Exception as e:
            logger.error(f"❌ Enhancer node failed: {e}")
            return Command(
                update={'messages': [HumanMessage(content=f"Enhancer error fallback: {e}", name='enhancer')]},
                goto='supervisor'
            )

    def research_node(self, state: MessagesState) -> Command[Literal['validator']]:
        try:
            research_agent = create_react_agent(
                model=self.llm,
                tools=[self.tools.tavily],
            )
            agent_state = {
                'messages': [
                    {'role': 'system', 'content': self.config.researcher_state_modifier}
                ] + state['messages']
            }
            res = research_agent.invoke(agent_state)
            last_msg = res['messages'][-1].content
            logger.debug(f"🔍 Researcher gathered info: {last_msg}")
            return Command(
                update={'messages': [HumanMessage(content=last_msg, name='researcher')]},
                goto='validator'
            )
        except Exception as e:
            logger.error(f"❌ Researcher node failed: {e}")
            return Command(
                update={'messages': [HumanMessage(content=f"Researcher error fallback: {e}", name='researcher')]},
                goto='validator'
            )

    def coder_node(self, state: MessagesState) -> Command[Literal['validator']]:
        try:
            code_agent = create_react_agent(
                model=self.llm,
                tools=[self.tools.python_repl],
            )
            coder_state = {
                'messages': [
                    {'role': 'system', 'content': self.config.coder_state_modifier}
                ] + state['messages']
            }
            res = code_agent.invoke(coder_state)
            last_msg = res['messages'][-1].content
            logger.debug(f"💻 Coder produced output: {last_msg}")
            return Command(
                update={'messages': [HumanMessage(content=last_msg, name='coder')]},
                goto='validator'
            )
        except Exception as e:
            logger.error(f"❌ Coder node failed: {e}")
            return Command(
                update={'messages': [HumanMessage(content=f"Coder error fallback: {e}", name='coder')]},
                goto='validator'
            )

    def validator_node(self, state: MessagesState) -> Command[Literal['supervisor', '__end__']]:
        try:
            user_question = state['messages'][0].content
            agent_answer = state['messages'][-1].content
            messages = [
                {
                    'role': 'system',
                    'content': self.config.validator_prompt
                },
                {
                    'role': 'user',
                    'content': f"User question: {user_question}"
                },
                {
                    'role': 'user',
                    'content': f"Agent answer: {agent_answer}"
                }
            ]
            res = self.llm.with_structured_output(ValidatorDecision).invoke(messages)
            goto = res.next
            reason = res.reason
            if goto == 'FINISH':
                logger.debug(f"✅ Validator decided to FINISH & reason: {reason}")
                goto = END
            else:
                logger.debug(f"🔄 Validator routed back to supervisor & reason: {reason}")

            return Command(
                update={'messages': [HumanMessage(content=f"Validator routed to {goto} because: {reason}", name='validator')]},
                goto=goto
            )
        except Exception as e:
            logger.error(f"❌ Validator node failed: {e}")
            return Command(
                update={'messages': [HumanMessage(content=f"Validator error fallback: {e}", name='validator')]},
                goto=END
            )

    # graph construction
    def _register_nodes(self) -> None:
        self.graph.add_node('supervisor', self.supervisor_node)
        self.graph.add_node('enhancer', self.enhancer_node)
        self.graph.add_node('researcher', self.research_node)
        self.graph.add_node('coder', self.coder_node)
        self.graph.add_node('validator', self.validator_node)
        logger.info("🔗 Nodes registered in the graph.")

    def _build_graph(self):
        self.graph.add_edge(START, 'supervisor')
        self.app = self.graph.compile()
        logger.info("🛠️ Graph built and compiled.")

    def visualize(self) -> None:
        try:
            graph_obj_getter = getattr(self.app, 'get_graph', None)
            graph_obj = graph_obj_getter() if callable(graph_obj_getter) else self._graph

            # 2. Mermaid source ---------------------------------------------------
            if hasattr(graph_obj, 'draw_mermaid'):
                try:
                    mermaid_src = graph_obj.draw_mermaid()
                    print(mermaid_src)
                    logger.info("🧪 Mermaid diagram (text) printed.")
                except Exception as e:
                    logger.debug("⚠️ Mermaid render not available: %s", e)

            # 3. ASCII fallback ---------------------------------------------------
            if hasattr(graph_obj, 'draw_ascii'):
                try:
                    ascii_map = graph_obj.draw_ascii()
                    print(ascii_map)
                    logger.info("📄 ASCII graph printed.")
                    return
                except Exception as e:
                    logger.debug("⚠️ ASCII render not available: %s", e)
        except Exception as e:
            logger.warning(f"❌ Graph visualization failed: {e}")
    
    def run(self, user_query: str, stream: bool = True) -> List[HumanMessage]:
        """
        Execute the workflow for a given user query.

        Args:
            user_query: Original user input.
            stream: Whether to stream intermediate events.

        Returns:
            List of final message objects.
        """
        inputs: Dict[str, Any] = {'messages': [('user', user_query)]}
        collected: List[HumanMessage] = []
        logger.info(f"🏁 Workflow started with query: {user_query}")

        if not stream:
            res = self.app.invoke(inputs)
            collected.extend(res['messages'])
            logger.debug(f"🎯 Workflow completed (non-stream) with \nres: {res}\n\ncollected: {collected}")
            return collected
        
        try:
            for e in self.app.stream(inputs):
                for node_name, node_state in e.items():
                    if not node_state:
                        continue
                    last_msg = node_state.get('messages', [])[-1]
                    collected.append(last_msg)
                    logger.debug(f"🔄 Stream update from node: {node_name}\n output: {last_msg}")
        except Exception as e:
            logger.error(f"❌ Streaming run failed: {e}")
            raise
        logger.info("🎯 Workflow completed (stream).")
        return collected

# helper functions

In [ ]:
def summarize_run(messages: List[HumanMessage]) -> str:
    """Produce a concise textual summary of the run (for debugging/reporting)."""
    summary_lines = []
    for m in messages:
        actor = m.name or "unknown"
        snippet = (m.content or "").replace("\n", " ")[:140]
        summary_lines.append(f"[{actor}] {snippet}")
    return "\n".join(summary_lines)

# main execution function

In [ ]:
def main(user_query: Optional[str] = None) -> None:
    query = user_query or DEFAULT_USER_QUERY
    config = WorkflowConfig()
    workflow = MultiAgentWorkflow(config)
    workflow.visualize()
    res = workflow.run(query, stream=True)
    summary = summarize_run(res)
    print("\n=== Run Summary ===")
    print(f"You: {query}")
    print(f"Assistant: {res[-2].content}")
    print(summary)
    print("=======================\n")

In [ ]:
main('calculate factorial of 5')